# Alertas y SRE Básico

SLOs, SLAs, reglas de alerta y respuesta a incidentes

## Introducción

Una alerta útil indica impacto, umbral y una acción o runbook. No es solo un número — debe decir qué está pasando, qué tan grave es, y qué hacer al respecto.

### Objetivos de Aprendizaje

- Entender qué característica tiene una alerta útil
- Definir SLOs y SLAs correctamente
- Crear reglas de alerta accionables
- Implementar runbooks para respuesta a incidentes
- Reducir alertas sin acción (alert fatigue)

## Qué característica tiene una alerta útil

> Una alerta útil indica impacto (quién se ve afectado), umbral (cuándo dispara), y acción o runbook (qué hacer). Sin estos tres elementos, la alerta genera ruido sin resolver problemas.

In [ ]:
print("=== Alerta Útil ===")

alerta_util = {
    "Impacto": "Quién se ve afectado y qué se pierde",
    "Umbral": "Condición específica que dispara la alerta",
    "Accion/Runbook": "Qué hacer para resolver o investigar",
}

for clave, valor in alerta_util.items():
    print(f"  {clave}: {valor}")

print("""
Ejemplo de alerta útil:

Alert: API High Latency
Impacto: Usuarios no pueden completar checkout
Umbral: p99 > 2s por 5 minutos
Accion: Verificar pods de checkout, revisar logs en Kibana
Runbook: https://wiki.internal/runbooks/checkout-latency

Esto permite actuar inmediatamente sin investigación adicional.
""")

## SLOs y SLAs: Definiciones

> SLO (Service Level Objective) es el objetivo interno de disponibilidad. SLA (Service Level Agreement) es el compromiso contractual con el cliente. El SLA es más permisivo que el SLO.

In [ ]:
print("=== SLOs vs SLAs ===")

comparacion = {
    "SLO (Service Level Objective)": {
        "Proposito": "Objetivo interno de disponibilidad",
        "Ejemplo": "99.9% de uptime mensual",
        "Quien lo define": "Equipo de ingeniería",
        "Consecuencia": "Alertas internas antes de violar SLA",
    },
    "SLA (Service Level Agreement)": {
        "Proposito": "Contrato con el cliente",
        "Ejemplo": "99.5% de uptime mensual",
        "Quien lo define": "Legal/comercial",
        "Consecuencia": "Penalizaciones económicas",
    },
}

for nombre, info in comparacion.items():
    print(f"\n{nombre}:")
    for k, v in info.items():
        print(f"  {k}: {v}")

print("""
Cálculo de downtime permitido:
  SLO 99.9% = 43.8 min/mes downtime
  SLO 99.95% = 21.9 min/mes downtime
  SLO 99.99% = 4.4 min/mes downtime

Error Budget = 100% - SLO
Para 99.9%: 0.1% del mes = budget para incidentes planificados
""")

## Reglas de Alerta Acciónables

> Cada alerta debe tener: condición precisa, ventana de tiempo, severidad correcta, y acción asociada. Evita alertas basadas en umbrales arbitrarios sin contexto.

In [ ]:
print("=== Reglas de Alerta ===")

reglas = [
    {
        "nombre": "High Error Rate",
        "condicion": "rate(http_requests_total{status=~'5..'}[5m]) > 0.01",
        "for": "5m",
        "severidad": "critical",
        "impacto": "Usuarios ven errores 500",
        "accion": "Revisar logs, verificar DB, hacer rollback si necesario",
    },
    {
        "nombre": "High Latency",
        "condicion": "histogram_quantile(0.99, rate(http_request_duration_seconds[5m])) > 2",
        "for": "5m",
        "severidad": "warning",
        "impacto": "Usuarios experimentan lentitud",
        "accion": "Verificar recursos, buscar queries lentas",
    },
    {
        "nombre": "Low Disk Space",
        "condicion": "node_filesystem_avail_bytes / node_filesystem_size_bytes < 0.1",
        "for": "10m",
        "severidad": "warning",
        "impacto": "Riesgo de servicio detenido",
        "accion": "Limpiar logs, expandir volumen",
    },
]

for regla in reglas:
    print(f"\n{regla['nombre']} ({regla['severidad']}):")
    print(f"  Condición: {regla['condicion']}")
    print(f"  For: {regla['for']}")
    print(f"  Impacto: {regla['impacto']}")
    print(f"  Acción: {regla['accion']}")

## Runbooks para Respuesta a Incidentes

> Un runbook es un documento que describe paso a paso cómo resolver una alerta específica. Debe existir ANTES de que la alerta dispare, no durante el incidente.

In [ ]:
runbook_template = """
# Runbook: High API Latency

## Impacto
Usuarios experimentan tiempos de respuesta > 2s en /api/checkout

## Verificación Inicial
1. Confirmar que el problema es real en Grafana
2. Identificar desde cuándo ocurre (breve vs sostenido)
3. Verificar si hay alertas relacionadas (DB, pods)

## Diagnóstico
1. Revisar panel de latencia por endpoint:
   http_request_duration_seconds{endpoint='/api/checkout'}

2. Verificar uso de recursos en pods:
   kubectl top pods -l app=checkout

3. Revisar logs por errores o timeouts:
   kubectl logs -l app=checkout --tail=100 | grep -i error

4. Verificar conexión a DB:
   psql -h db.internal -c "SELECT count(*) FROM pg_stat_activity;"

## Resolución
1. Si DB saturada: matar queries lentas
2. Si pods faltantes: kubectl scale deployment checkout --replicas=5
3. Si memory pressure: restart pods kubectl rollout restart deployment/checkout

## Escalación
Si no se resuelve en 15 min, escalar a @oncall-senior
"""

print("=== Runbook Template ===")
print(runbook_template)

## Niveles de Severidad

> Define severidades claras: Critical (servicio caído, acción inmediata), Warning (degradación, investigar pronto), Info (informativo, no requiere acción inmediata).

In [ ]:
print("=== Niveles de Severidad ===")

severidades = {
    "SEV1 - Critical": {
        "Ejemplos": "Servicio caído, data loss, security breach",
        "Tiempo respuesta": "15 minutos",
        "Canales": "Phone call, SMS, PagerDuty",
        "Accion": "Resolución inmediata, todos manos disponibles",
    },
    "SEV2 - High": {
        "Ejemplos": "Degradación severa, funcionalidad crítica afectada",
        "Tiempo respuesta": "1 hora",
        "Canales": "Slack #incidents, PagerDuty",
        "Accion": "On-call responde, equipo se coordina",
    },
    "SEV3 - Warning": {
        "Ejemplos": "Métricas degradas, funcionalidad secundaria afectada",
        "Tiempo respuesta": "4 horas",
        "Canales": "Slack #monitoring",
        "Accion": "On-call revisa en siguiente día laborable",
    },
    "SEV4 - Info": {
        "Ejemplos": "Logs inusuales, métricas ligeramente fuera de baseline",
        "Tiempo respuesta": "Next sprint",
        "Canales": "Ticket system",
        "Accion": "Registrar y trackear, no requiere respuesta inmediata",
    },
}

for sev, info in severidades.items():
    print(f"\n{sev}:")
    for k, v in info.items():
        print(f"  {k}: {v}")

## Reducir Alert Fatigue

> Alert fatigue ocurre cuando hay demasiadas alertas sin acción. El equipo deja de prestar atención. Soluciones: alertas con for apropiado, deduplicación, y eliminar alertas que no llevan a acción.

In [ ]:
print("=== Alert Fatigue ===")

print("""
Causas de alert fatigue:
  - Alertas que no llevan a acción (no se puede hacer nada)
  - Spikes transitorios disparan alertas sin ser problema real
  - Multiples alertas para el mismo problema (no deduplicadas)
  - Umbrales muy sensibles sin justificación

Soluciones:
""")

soluciones = {
    "Usa 'for'": "Espera 5m antes de disparar, evita spikes",
    "Elimina sin acción": "Si no hay runbook, no debe ser alerta",
    "Deduplica": "Una alerta por problema, no por componente",
    "Ajusta umbrales": "Usa datos históricos para umbrales realistas",
    "Usa SLO burn": "Alerta cuando el error budget se está consumiendo",
}

for sol, desc in soluciones.items():
    print(f"  {sol}: {desc}")

## Flujo de Respuesta a Incidentes

> El flujo típico: detección → triaje → mitigación → resolución → postmortem. La clave es minimizar MTTR (Mean Time To Resolve) con runbooks y comunicación clara.

In [ ]:
print("=== Flujo de Incidente ===")

flujo = [
    ("1. Detección", "Alerta dispara o usuario reporta", "¿Está confirmado?"),
    ("2. Triaje", "On-call investiga causa raíz", "¿Qué está roto?"),
    ("3. Mitigación", "Acción inmediata para reducir impacto", "¿Podemos restaurar servicio?"),
    ("4. Resolución", "Fix permanente desplegado", "¿Está resuelto?"),
    ("5. Postmortem", "Documentar, aprender, mejorar", "¿Cómo evitar repetición?"),
]

for paso, desc, pregunta in flujo:
    print(f"{paso}: {desc}")
    print(f"  Pregunta clave: {pregunta}\n")

print("""
MTTR (Mean Time To Resolve):
  - Medir tiempo desde alerta hasta resolución
  - Meta: reducir progresivamente
  - Si MTTR > 1h, revisar si runbooks necesitan mejora
""")

## Alertas Basadas en SLOs

> Las mejores alertas miden el burn rate del error budget. Si el budget se consume demasiado rápido, hay que actuar antes de violar el SLO.

In [ ]:
print("=== Alertas de Error Budget ===")

slo_alert = """
SLO: 99.9% mensual (0.1% error budget)
Budget mensual: 43.8 minutos de downtime

Alert: Budget Burn Rate Critical
Condición: error_budget / total_budget < 0.25 after 1h
Interpretación: Queda menos de 25% del budget

Alert: Budget Burn Rate Warning  
Condición: error_budget / total_budget < 0.5 after 1h
Interpretación: Queda menos de 50% del budget
"""

print(slo_alert)

print("""
PromQL para error budget:
  # Budget consumido
  sum(error_budget_consumed{slo="api-availability"})

  # Burn rate (cuánto budget se consume por hora)
  sum(rate(http_requests_total{status=~"5.."}[1h]))
    /
  sum(rate(http_requests_total[1h]))

Beneficio: Alertas proactivas antes de violar SLO, no después.
""")

## Tips y Mejores Prácticas

> Cada alerta debe tener un runbook. Si no puedes escribir qué hacer, no es una alerta — es información.

> El 'for' es tu amigo: 5m de condición sostenida evita disparar por spikes. Ajusta según tolerancia.

> Prioriza alertas que miden impacto en usuario, no solo uso de recursos (CPU puede estar alto sin afectar usuarios).

> Haz postmortems de cada SEV1 y SEV2. La pregunta no es quién fault, sino cómo evitar que ocurra.

> Revisa tus alertas trimestralmente. Elimina las que no dispararon acción en 3 meses.

## Errores Comunes

### Alertas sin acción

¿Por qué ocurre?
- Se alerta "CPU > 80%" pero nadie sabe qué hacer con eso.

Solución
- Si no hay runbook, no debe ser alerta. Cambia a información o elimina.

### Alertas sin 'for'

¿Por qué ocurre?
- Un spike de 10 segundos dispara alerta aunque no sea problema real.

Solución
- Usa 'for: 5m' para requerir condición sostenida antes de notificar.

### No medir MTTR

¿Por qué ocurre?
- No se sabe si los runbooks están funcionando.

Solución
- Registra tiempo de cada incidente. Si MTTR sube, los runbooks necesitan mejora.

### Umbrales arbitrarios

¿Por qué ocurre?
- Se pone "latency > 1s" porque alguien lo pensó razonable.

Solución
- Usa datos históricos (p95 actual) + margen razonable. No números inventados.

### Postmortems con blame

¿Por qué ocurre?
- El equipo teme admitir errores y los postmortems se vuelven defensivos.

Solución
- Postmortems son blameless. El foco es proceso y sistema, no personas.